In [ ]:
!python -m spacy download es_core_news_sm
!pip install spacy[transformers]
!pip install mglearn
!python -m spacy download es_dep_news_trf
!pip install mglearn
!pip install pyLDAvis

<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

import pandas as pd
import spacy
from google.colab import drive
from nltk import word_tokenize
from nltk import SnowballStemmer
import nltk
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import numpy as np
import mglearn
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim import corpora, models
from pprint import pprint
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)
from sklearn.manifold import TSNE
from bokeh.plotting import figure, output_file, show
from bokeh.models import Label
from bokeh.io import output_notebook
import matplotlib.colors as mcolors
from bokeh.core.enums import MarkerType
import seaborn as sns
sns.set_style("darkgrid")
%matplotlib inline
nltk.download('punkt')

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

## Importando tweets 

In [ ]:
# ================== OJO modificar para cada serie ==================

tweets = pd.ExcelFile('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/\
Los80/19-05-2022 13.35.30_NO_DUPLICATES.xlsx')

# ================== OJO modificar para cada serie ==================
tweets

In [ ]:
print(tweets.sheet_names)

In [ ]:
df=tweets.parse('Sheet1')

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df=df[df.text.notnull()]
df

## Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos) 

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
df['tweetModificado']=df['text'].apply(codigohex)

In [ ]:
df['tweetModificado']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropunicode)

In [ ]:
df['tweetModificado']

##### Eliminar secuencias de escape

In [ ]:
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\r', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\n', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\v', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\t', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\f', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('\b', ' '))
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('80', 'ochenta')) # serie los 80
df['tweetModificado'] = df['tweetModificado'].apply(str.replace, args=('ano', 'año')) # serie los 80

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropURLs)
df['tweetModificado']=df['tweetModificado'].apply(dropURL)

In [ ]:
df['tweetModificado']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropemoticons)

In [ ]:
df['tweetModificado']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropemails)

In [ ]:
df['tweetModificado']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropmenciones)

In [ ]:
df['tweetModificado']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(droppalabrasrepetidas)

In [ ]:
df['tweetModificado']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropnumeros)

In [ ]:
df['tweetModificado']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation) 
punct 

In [ ]:
punct.remove('#')

In [ ]:
punct

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(puntuacion)

In [ ]:
df['tweetModificado']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropespacios)

In [ ]:
df['tweetModificado']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(droppalabrascortas)

In [ ]:
df['tweetModificado']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(normatext)

In [ ]:
df['tweetModificado']

##### Eliminando stopwords

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import WhitespaceTokenizer
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
swsp=nltk.corpus.stopwords.words('spanish')

In [ ]:
swpo=nltk.corpus.stopwords.words('portuguese')

In [ ]:
swen=nltk.corpus.stopwords.words('english')

In [ ]:
def dropStopWordsSpanish(texto):
  if (str(texto) != 'nan'):
    tk = WhitespaceTokenizer()
    tokens=tk.tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swpo:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsPortugues(texto):
  if (str(texto) != 'nan'):
    tk = WhitespaceTokenizer()
    tokens=tk.tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsEnglish(texto):
  if (str(texto) != 'nan'):
    tk = WhitespaceTokenizer()
    tokens=tk.tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swen:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropStopWordsSpanish)
df['tweetModificado']=df['tweetModificado'].apply(dropStopWordsPortugues)
df['tweetModificado']=df['tweetModificado'].apply(dropStopWordsEnglish)

In [ ]:
df['tweetModificado']

##### Volvemos a eliminar exceso de espacios

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(dropespacios)

In [ ]:
df['tweetModificado']

##### Volvemos a eliminar tokens cortos

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(droppalabrascortas)

In [ ]:
df['tweetModificado']

##### Lematizar

In [ ]:
nlp = spacy.load("es_dep_news_trf")
nlp

In [ ]:
nlp.vocab

In [ ]:
from spacy.tokenizer import Tokenizer

In [ ]:
nlp.tokenizer = Tokenizer(nlp.vocab, token_match=re.compile(r'\S+').match)

In [ ]:
df['tweetModificado']

In [ ]:
def lematizarprofundo(texto):
  doc_spacy = nlp(texto)
  tokens_lema= " ".join([token.lemma_ for token in doc_spacy])
  return tokens_lema

In [ ]:
df['tweetModificado']=df['tweetModificado'].apply(lematizarprofundo)

In [ ]:
df['tweetModificado']

## Data

In [ ]:
df

## Exportamos datos

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/Los80/'

# ======================= OJO Cambiar PATH para cada Serie =======================

In [ ]:
df.to_excel(PATH_TO_SAVE_SEARCHES+'RESULTS-tweets-Cleaning.xlsx', index=False)

## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
!pip install Counter
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

import nltk
from nltk import word_tokenize
from nltk.probability import FreqDist
from matplotlib import pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from numpy import dot
from numpy.linalg import norm

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

## Tweets

### Importando datos

In [ ]:
# ================== OJO modificar para cada serie ==================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/TwitterResults/\
Los80/RESULTS-tweets-Cleaning.xlsx' 

# ================== OJO modificar para cada serie ==================

Tweets = pd.ExcelFile(path)
Tweets

In [ ]:
Tweets=Tweets.parse('Sheet1')

In [ ]:
Tweets.info()

In [ ]:
Tweets=Tweets[Tweets.tweetModificado.notnull()]
Tweets

### Tokenizamos Comentarios

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.tokenize import WhitespaceTokenizer

In [ ]:
def tokens(texto):
  tk = WhitespaceTokenizer()
  texto_tokens=tk.tokenize(texto)
  return texto_tokens

In [ ]:
Tweets['tweetTokenizado']=Tweets['tweetModificado'].apply(tokens)

In [ ]:
Tweets

### Construimos gramas, bigramas y trigramas

In [ ]:
def bigrama(lista):
  bigrama=list(nltk.ngrams(lista, 2))
  return bigrama

def trigrama(lista):
  trigrama=list(nltk.ngrams(lista, 3))
  return trigrama

In [ ]:
sentenceTotalTweets =" ".join(Tweets['tweetModificado'])
tk = WhitespaceTokenizer()
tokensTotalTweets = tk.tokenize(sentenceTotalTweets)
print(len(tokensTotalTweets))
Tweets['bigramas']=Tweets['tweetTokenizado'].apply(bigrama)
Tweets['trigramas']=Tweets['tweetTokenizado'].apply(trigrama)

In [ ]:
Tweets

### Wordcloud de Gramas, Bigramas y Trigramas

In [ ]:
datos='tweets' # 'Comentarios' 
serie='Los80' #
tv='Los 80'
#colormap='inferno->Narcos, colormap='viridis->EPM, colormap='cividis->Tres Caines, colormap='magma'->Dignity, colormap='plasma->Los80
#palette='dark:salmon_r'->Narcos, palette='ch:s=-.2,r=.6'->EPM, palette='coolwarm'->Tres Caines, palette='rocket_r'->Dignity, palette='crest'->Los80

In [ ]:
def wordcloud(columna):
  l=list(columna)
  oracion=[]
  for i in range(len(l)):
    for j in range(len(l[i])):
      oracion.append('_'.join(l[i][j]))
  return oracion

In [ ]:
TokensBigramas=wordcloud(Tweets['bigramas'])

In [ ]:
TokensTrigramas=wordcloud(Tweets['trigramas'])

In [ ]:
OracionGramas=" ".join(tokensTotalTweets)
OracionGramas

In [ ]:
wordcloud = WordCloud(width=2000, height=1500, contour_width=1, contour_color='black', scale=3, min_font_size=5, max_font_size=500, max_words=500, background_color='white', relative_scaling=0.5, collocations=False, colormap='inferno', normalize_plurals=True, repeat=False, include_numbers=False, min_word_length=4, regexp=r'\S+').generate(OracionGramas)
#plot the wordcloud
plt.figure(figsize = (15, 15))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Nube de palabras (500) para tweets. \nSerie {}'.format(tv), fontsize=20)
#to remove the axis value
plt.axis("off")
plt.show()

In [ ]:
wordcloud.to_file("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/WordCloudPalabras"+datos+tv+".png")

In [ ]:
OracionBigramas=" ".join(TokensBigramas)
OracionBigramas

In [ ]:
wordcloud = WordCloud(width=2000, height=1500, contour_width=1, contour_color='black', scale=3, min_font_size=5, max_font_size=500, max_words=500, background_color='white', relative_scaling=0.5, collocations=False, colormap='inferno', normalize_plurals=True, repeat=False, include_numbers=False, min_word_length=7, regexp=r'\S+').generate(OracionBigramas)
#plot the wordcloud
plt.figure(figsize = (15, 15))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Nube de bigramas (500) para tweets. \nSerie {}'.format(tv), fontsize=20)
#to remove the axis value
plt.axis("off")
plt.show()

In [ ]:
wordcloud.to_file("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/WordCloudBigramas"+datos+tv+".png")

In [ ]:
OracionTrigramas=" ".join(TokensTrigramas)
OracionTrigramas

In [ ]:
wordcloud = WordCloud(width=2000, height=1500, contour_width=1, contour_color='black', scale=3, min_font_size=5, max_font_size=500, max_words=500, background_color='white', relative_scaling=0.5, collocations=False, colormap='inferno', normalize_plurals=True, repeat=False, include_numbers=False, min_word_length=14, regexp=r'\S+').generate(OracionTrigramas)
#plot the wordcloud
plt.figure(figsize = (15, 15))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Nube de trigramas (500) para tweets. \nSerie {}'.format(tv), fontsize=20)
#to remove the axis value
plt.axis("off")
plt.show()

In [ ]:
wordcloud.to_file("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/WordCloudTrigramas"+datos+tv+".png")

### Flaten de lista de bigramas y trigramas

In [ ]:
b=[]
for i in range(len(Tweets)):
  b.append(Tweets['bigramas'].iloc[i])
flat_bigramas = [item for sublist in b for item in sublist]

In [ ]:
len(flat_bigramas)

In [ ]:
t=[]
for i in range(len(Tweets)):
  t.append(Tweets['trigramas'].iloc[i])
flat_trigramas = [item for sublist in t for item in sublist]

In [ ]:
len(flat_trigramas)

### Conteo de palabras, bigramas y trigramas

In [ ]:
datos='Tweets' # 
serie='Los80' #
tv='Los 80'

#### Palabras

In [ ]:
counted = Counter(tokensTotalTweets)
word_freqTotalTextDesc = pd.DataFrame(counted.items(),columns=['Palabra','Frecuencia'] ).sort_values(by='Frecuencia',ascending=False, ignore_index=True)
word_freqTotalTextAsc = pd.DataFrame(counted.items(),columns=['Palabra','Frecuencia'] ).sort_values(by='Frecuencia',ascending=True, ignore_index=True)

In [ ]:
len(counted)

In [ ]:
word_freqTotalTextDesc

In [ ]:
indice=[]
for i in range(len(word_freqTotalTextDesc)):
  palabra=word_freqTotalTextDesc['Palabra'].iloc[i]
  longitud=int(len(palabra))
  if (longitud <= 3):
    indice.append(i)

In [ ]:
word_freqTotalTextDesc.drop(indice , inplace=True)
word_freqTotalTextDesc.head(30)

In [ ]:
word_freqTotalTextAsc

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Mayor numero de frecuencia de palabras (30) para tweets. \nSerie {}".format(tv))
grafica1=sns.barplot(x='Frecuencia',y='Palabra',data=word_freqTotalTextDesc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica1.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MayorFrecuenciaPalabras"+datos+tv+".png",bbox_inches='tight')

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Menor numero de frecuencia de palabras (30) para tweets. \nSerie {}".format(tv))
grafica2=sns.barplot(ax=axes,x='Frecuencia',y='Palabra',data=word_freqTotalTextAsc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica2.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MenorFrecuenciaPalabras"+datos+tv+".png",bbox_inches='tight')

#### Bigramas

In [ ]:
counted = Counter(flat_bigramas)
word_freqTotalTextDesc = pd.DataFrame(counted.items(),columns=['Bigrama','Frecuencia'] ).sort_values(by='Frecuencia',ascending=False, ignore_index=True)
word_freqTotalTextAsc = pd.DataFrame(counted.items(),columns=['Bigrama','Frecuencia'] ).sort_values(by='Frecuencia',ascending=True, ignore_index=True)

In [ ]:
len(counted)

In [ ]:
word_freqTotalTextDesc

In [ ]:
indice=[]
for i in range(len(word_freqTotalTextDesc)):
  palabra1,palabra2=word_freqTotalTextDesc['Bigrama'].iloc[i]
  longitud=int(len(palabra1+palabra2))
  if (longitud <= 6):
    indice.append(i)

In [ ]:
word_freqTotalTextDesc.drop(indice , inplace=True)
word_freqTotalTextDesc.head(30)

In [ ]:
word_freqTotalTextAsc

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Mayor numero de frecuencia de bigramas (30) para tweets. \nSerie {}".format(tv))
grafica3=sns.barplot(x='Frecuencia',y='Bigrama',data=word_freqTotalTextDesc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica3.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MayorFrecuenciaBigramas"+datos+tv+".png" ,bbox_inches='tight')

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Menor numero de frecuencia de bigramas (30) para tweets. \nSerie {}".format(tv))
grafica4=sns.barplot(ax=axes,x='Frecuencia',y='Bigrama',data=word_freqTotalTextAsc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica4.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MenorFrecuenciaBigramas"+datos+tv+".png",bbox_inches='tight')

#### Trigramas

In [ ]:
counted = Counter(flat_trigramas)
word_freqTotalTextDesc = pd.DataFrame(counted.items(),columns=['Trigrama','Frecuencia'] ).sort_values(by='Frecuencia',ascending=False, ignore_index=True)
word_freqTotalTextAsc = pd.DataFrame(counted.items(),columns=['Trigrama','Frecuencia'] ).sort_values(by='Frecuencia',ascending=True, ignore_index=True)

In [ ]:
len(counted)

In [ ]:
word_freqTotalTextDesc

In [ ]:
indice=[]
for i in range(len(word_freqTotalTextDesc)):
  palabra1,palabra2,palabra3=word_freqTotalTextDesc['Trigrama'].iloc[i]
  longitud=int(len(palabra1+palabra2+palabra3))
  if (longitud <= 13):
    indice.append(i)

In [ ]:
word_freqTotalTextDesc.drop(indice , inplace=True)
word_freqTotalTextDesc.head(30)

In [ ]:
word_freqTotalTextAsc

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Mayor numero de frecuencia de trigramas (30) para tweets. \nSerie {}".format(tv))
grafica5=sns.barplot(x='Frecuencia',y='Trigrama',data=word_freqTotalTextDesc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica5.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MayorFrecuenciaTrigramas"+datos+tv+".png",bbox_inches='tight')

In [ ]:
sns.set_style("darkgrid")
fig, axes = plt.subplots(figsize=(12,10))
plt.title("Menor numero de frecuencia de trigramas (30) para tweets. \nSerie {}".format(tv))
grafica6=sns.barplot(ax=axes,x='Frecuencia',y='Trigrama',data=word_freqTotalTextAsc.head(30), palette='dark:salmon_r')
plt.show()

In [ ]:
grafica6.figure.savefig("/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/DOCUMENTS/TwitterReport/"+serie+"/MenorFrecuenciaTrigramas"+datos+tv+".png",bbox_inches='tight')